In [4]:
# Install required packages
!pip install -q transformers datasets accelerate peft bitsandbytes trl
!pip install -q gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.7 MB/s eta 0:00:00


In [5]:
import huggingface_hub
huggingface_hub.login()

In [7]:
from datasets import load_dataset

# Load the dataset
dataset_name = "gretelai/synthetic_text_to_sql"
dataset = load_dataset(dataset_name, split="train")

# Use different seed and slightly larger sample
dataset = dataset.shuffle(seed=2024).select(range(1000))

# Keep 80/20 split but with new seed
dataset = dataset.train_test_split(test_size=0.2, seed=2024)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

# Simplified prompt format (still works with the model)
def format_prompt(example):
    """
    Creates a formatted prompt for fine-tuning.
    """
    prompt = f"""### Task:
Generate a SQL query based on the given schema and question.

### Schema:
{example['sql_context']}

### Question:
{example['sql_prompt']}

### Answer:
SQL: {example['sql']}
Explanation: {example['sql_explanation']}
"""
    return {"text": prompt}

# Apply formatting
train_dataset = train_dataset.map(format_prompt)
test_dataset = test_dataset.map(format_prompt)

print(f"Training examples: {len(train_dataset)}")
print(f"Testing examples: {len(test_dataset)}")

Training examples: 800
Testing examples: 200


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# Modified quantization config (still uses 4-bit but different settings)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,  # Changed to False
)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model and tokenizer loaded successfully!")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Model and tokenizer loaded successfully!


In [10]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Modified LoRA config with different rank
lora_config = LoraConfig(
    r=8,  # Changed from 16 to 8 (lower rank, faster training)
    lora_alpha=16,  # Changed from 32
    lora_dropout=0.1,  # Changed from 0.05
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]  # Keep same modules for compatibility
)

# Prepare model for k-bit training and add LoRA adapters
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

Trainable parameters: 3,407,872 (0.09%)


In [11]:
from transformers import TrainingArguments
from trl import SFTTrainer

# Modified training arguments
training_args = TrainingArguments(
    output_dir="./sql_model_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_8bit",
    learning_rate=3e-4,
    lr_scheduler_type="cosine",
    logging_steps=20,
    report_to="none",
    save_strategy="epoch",
    warmup_steps=10,
)

# Initialize the SFTTrainer (FIXED - removed problematic parameters)
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=lora_config,
    args=training_args,
    # Removed: dataset_text_field, max_seq_length (these cause the error)
)

print("=" * 70)
print("STARTING FINE-TUNING")
print("=" * 70)
trainer.train()
print("=" * 70)
print("FINE-TUNING COMPLETE")
print("=" * 70)

# Save the model
trainer.save_model("sql-mistral-finetuned")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

STARTING FINE-TUNING


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,0.806700
40,0.525700
60,0.483100
80,0.461400
100,0.467000
120,0.443600
140,0.442700
160,0.442300
180,0.427500
200,0.427600


FINE-TUNING COMPLETE


In [12]:
from peft import PeftModel
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Load tokenizer
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Load test data
from datasets import load_dataset
dataset = load_dataset("gretelai/synthetic_text_to_sql", split="train")
dataset = dataset.shuffle(seed=2024).select(range(1000))
dataset = dataset.train_test_split(test_size=0.2, seed=2024)
test_dataset = dataset["test"]

# Use different test example
test_example = test_dataset[10]  # Changed index

# Modified prompt template
prompt_template = """### Task:
Generate a SQL query based on the given schema and question.

### Schema:
{sql_context}

### Question:
{natural_language_question}

### Answer:
"""

inference_prompt = prompt_template.format(
    sql_context=test_example['sql_context'],
    natural_language_question=test_example['sql_prompt']
)

print("\n" + "=" * 70)
print("BASE MODEL OUTPUT")
print("=" * 70)
inputs_before = tokenizer(inference_prompt, return_tensors="pt").to("cuda")
outputs_before = base_model.generate(**inputs_before, max_new_tokens=256, use_cache=True)
decoded_output_before = tokenizer.decode(outputs_before[0], skip_special_tokens=True)
print(decoded_output_before)

# Load fine-tuned model
fine_tuned_model = PeftModel.from_pretrained(base_model, "sql-mistral-finetuned")

print("\n" + "=" * 70)
print("FINE-TUNED MODEL OUTPUT")
print("=" * 70)
inputs_after = tokenizer(inference_prompt, return_tensors="pt").to("cuda")
outputs_after = fine_tuned_model.generate(**inputs_after, max_new_tokens=256, use_cache=True)
decoded_output_after = tokenizer.decode(outputs_after[0], skip_special_tokens=True)
print(decoded_output_after)

print("\n" + "=" * 70)
print("COMPARISON COMPLETE")
print("=" * 70)

# Cleanup
del base_model
torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



BASE MODEL OUTPUT


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


### Task:
Generate a SQL query based on the given schema and question.

### Schema:
CREATE TABLE Exhibitions (id INT, name VARCHAR(20), start_date DATE, end_date DATE); INSERT INTO Exhibitions VALUES (1, 'Exhibition A', '2022-01-01', '2022-03-31'), (2, 'Exhibition B', '2022-02-01', '2022-04-30'), (3, 'Exhibition C', '2022-03-01', '2022-05-31'); CREATE TABLE Visitors (id INT, exhibition_id INT, visit_date DATE); INSERT INTO Visitors VALUES (1, 1, '2022-01-02'), (2, 1, '2022-01-03'), (3, 2, '2022-02-05'), (4, 3, '2022-03-07'), (5, 3, '2022-03-08');

### Question:
What is the average number of visitors per day for each exhibition?

### Answer:
```sql
SELECT e.name, AVG(v.num_visitors_per_day) as avg_visitors_per_day
FROM Exhibitions e
JOIN (
    SELECT exhibition_id, DATE(visit_date) as visit_date, COUNT(*) as num_visitors_per_day
    FROM Visitors
    GROUP BY exhibition_id, visit_date
) v ON e.id = v.exhibition_id
GROUP BY e.id;
```
Note: This query assumes that there is a `num_visitors

In [13]:
import random
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import re
import gc

# Cleanup memory
try:
    del fine_tuned_model
except:
    pass
try:
    del base_model
except:
    pass
try:
    del model
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print("=" * 70)
print("MODEL EVALUATION")
print("=" * 70)

# Helper function to extract SQL
def extract_sql(text):
    """Extract SQL from model output"""
    # Look for SQL after "Answer:" or "SQL:"
    patterns = [
        r'SQL:\s*(.*?)(?:\n\nExplanation:|Explanation:|$)',
        r'Answer:\s*SQL:\s*(.*?)(?:\n|$)',
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        if match:
            sql = match.group(1).strip()
            # Remove explanation if it got included
            sql = sql.split('Explanation:')[0].strip()
            return sql

    # Fallback: return first line that looks like SQL
    lines = text.split('\n')
    for line in lines:
        if any(kw in line.upper() for kw in ['SELECT', 'INSERT', 'UPDATE', 'DELETE']):
            return line.strip()

    return text.strip()

def normalize_sql(sql):
    """Normalize SQL for comparison"""
    sql = ' '.join(sql.split())
    sql = sql.replace(';', '').strip()
    return sql.upper()

def calculate_similarity(pred, truth):
    """Simple similarity calculation"""
    pred_norm = normalize_sql(pred)
    truth_norm = normalize_sql(truth)

    # Token-based similarity
    pred_tokens = set(pred_norm.split())
    truth_tokens = set(truth_norm.split())

    if len(truth_tokens) == 0:
        return 0.0

    intersection = len(pred_tokens & truth_tokens)
    union = len(pred_tokens | truth_tokens)

    return intersection / union if union > 0 else 0.0

# Select test examples
num_examples = 5
random.seed(999)  # Different seed
test_indices = random.sample(range(len(test_dataset)), num_examples)

test_cases = []
for idx in test_indices:
    example = test_dataset[idx]
    test_cases.append({
        'question': example['sql_prompt'],
        'ground_truth': example['sql'],
        'schema': example['sql_context'],
        'prompt': prompt_template.format(
            sql_context=example['sql_context'],
            natural_language_question=example['sql_prompt']
        )
    })

print(f"Selected {len(test_cases)} test examples\n")

# Evaluate Base Model
print("=" * 70)
print("EVALUATING BASE MODEL")
print("=" * 70)

model_name = "mistralai/Mistral-7B-Instruct-v0.2"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

for i, tc in enumerate(test_cases, 1):
    print(f"[{i}/{len(test_cases)}] Processing...")

    inputs = tokenizer(tc['prompt'], return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=256,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    sql = extract_sql(response.split('### Answer:')[-1])
    similarity = calculate_similarity(sql, tc['ground_truth'])

    tc['base_sql'] = sql
    tc['base_score'] = similarity

    torch.cuda.empty_cache()

print("Base model evaluation complete\n")

# Cleanup
del base_model
gc.collect()
torch.cuda.empty_cache()

# Evaluate Fine-tuned Model
print("=" * 70)
print("EVALUATING FINE-TUNED MODEL")
print("=" * 70)

base_for_ft = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

fine_tuned_model = PeftModel.from_pretrained(base_for_ft, "sql-mistral-finetuned")

for i, tc in enumerate(test_cases, 1):
    print(f"[{i}/{len(test_cases)}] Processing...")

    inputs = tokenizer(tc['prompt'], return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=256,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    sql = extract_sql(response.split('### Answer:')[-1])
    similarity = calculate_similarity(sql, tc['ground_truth'])

    tc['ft_sql'] = sql
    tc['ft_score'] = similarity
    tc['improvement'] = similarity - tc['base_score']

    torch.cuda.empty_cache()

print("Fine-tuned model evaluation complete\n")

# Display Results
print("=" * 70)
print("RESULTS")
print("=" * 70)

for i, tc in enumerate(test_cases, 1):
    status = "✓ IMPROVED" if tc['improvement'] > 0.05 else ("≈ SIMILAR" if abs(tc['improvement']) <= 0.05 else "✗ DEGRADED")

    print(f"\n{'=' * 70}")
    print(f"TEST CASE {i}")
    print(f"{'=' * 70}")
    print(f"Question: {tc['question']}")
    print(f"\nGround Truth:\n{tc['ground_truth']}")
    print(f"\nBase Model:\n{tc['base_sql']}")
    print(f"Score: {tc['base_score']:.1%}")
    print(f"\nFine-tuned Model:\n{tc['ft_sql']}")
    print(f"Score: {tc['ft_score']:.1%}")
    print(f"\n{status} (Change: {tc['improvement']:+.1%})")

# Summary
avg_base = sum(tc['base_score'] for tc in test_cases) / len(test_cases)
avg_ft = sum(tc['ft_score'] for tc in test_cases) / len(test_cases)
avg_improvement = avg_ft - avg_base

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Average Base Model Score: {avg_base:.1%}")
print(f"Average Fine-tuned Score: {avg_ft:.1%}")
print(f"Average Improvement: {avg_improvement:+.1%}")
print("=" * 70)

MODEL EVALUATION
Selected 5 test examples

EVALUATING BASE MODEL


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

[1/5] Processing...
[2/5] Processing...
[3/5] Processing...
[4/5] Processing...
[5/5] Processing...
Base model evaluation complete

EVALUATING FINE-TUNED MODEL


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

[1/5] Processing...
[2/5] Processing...
[3/5] Processing...
[4/5] Processing...
[5/5] Processing...
Fine-tuned model evaluation complete

RESULTS

TEST CASE 1
Question: What is the number of medical professionals per state?

Ground Truth:
SELECT s.state_name, mp.profession_count FROM states s JOIN medical_professionals mp ON s.state_id = mp.state_id;

Base Model:
SELECT states.state_name, medical_professionals.profession_count as num_professionals_per_state
Score: 5.9%

Fine-tuned Model:
SELECT states.state_name, medical_professionals.profession_count FROM states JOIN medical_professionals ON states.state_id = medical_professionals.state_id;
Score: 41.2%

✓ IMPROVED (Change: +35.3%)

TEST CASE 2
Question: How many satellites were launched by each country in the satellite_launches table?

Ground Truth:
SELECT country, COUNT(satellites) OVER (PARTITION BY country) FROM satellite_launches;

Base Model:
SELECT country, SUM(satellites) as total_satellites
Score: 16.7%

Fine-tuned Model:
SEL

In [ ]:
import gradio as gr
import re
import torch
import json
import csv
from datetime import datetime
import os

# Merge model for faster inference
try:
    merged_model = fine_tuned_model.merge_and_unload()
    model_for_demo = merged_model
    print("✅ Using merged model")
except:
    model_for_demo = fine_tuned_model
    print("⚠️ Using PEFT model")

torch.cuda.empty_cache()

# Create output directory for saving files
output_dir = "sql_generator_outputs"
os.makedirs(output_dir, exist_ok=True)

# Initialize history list
query_history = []

def generate_sql_response(schema, question):
    """
    Generate SQL query and explanation
    """
    prompt = f"""### Task:
Generate a SQL query based on the given schema and question.

### Schema:
{schema}

### Question:
{question}

### Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model_for_demo.generate(
        **inputs,
        max_new_tokens=300,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer_part = response.split('### Answer:')[-1].strip()

    # Parse SQL and explanation
    sql_match = re.search(r'SQL:\s*(.*?)(?:\n\n|\nExplanation:|$)', answer_part, re.DOTALL)
    exp_match = re.search(r'Explanation:\s*(.*?)$', answer_part, re.DOTALL)

    sql = sql_match.group(1).strip() if sql_match else answer_part.split('\n')[0]
    explanation = exp_match.group(1).strip() if exp_match else "Query generated successfully."

    # Save to history
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    history_entry = {
        "timestamp": timestamp,
        "schema": schema,
        "question": question,
        "sql": sql,
        "explanation": explanation
    }
    query_history.append(history_entry)

    return sql, explanation, f"✅ Query generated at {timestamp}"

def save_to_json():
    """Save all query history to JSON file"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{output_dir}/query_history_{timestamp}.json"

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(query_history, f, indent=2, ensure_ascii=False)

    return filename

def save_to_csv():
    """Save all query history to CSV file"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{output_dir}/query_history_{timestamp}.csv"

    with open(filename, 'w', newline='', encoding='utf-8') as f:
        if query_history:
            writer = csv.DictWriter(f, fieldnames=query_history[0].keys())
            writer.writeheader()
            writer.writerows(query_history)

    return filename

def save_current_query(schema, question, sql, explanation):
    """Save current query to individual file"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{output_dir}/query_{timestamp}.txt"

    content = f"""SQL QUERY GENERATED
{'='*60}
Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

DATABASE SCHEMA:
{schema}

QUESTION:
{question}

GENERATED SQL:
{sql}

EXPLANATION:
{explanation}
{'='*60}
"""

    with open(filename, 'w', encoding='utf-8') as f:
        f.write(content)

    return filename

def get_history_display():
    """Format history for display"""
    if not query_history:
        return "No queries generated yet."

    history_text = ""
    for i, entry in enumerate(reversed(query_history[-10:]), 1):  # Show last 10
        history_text += f"""
{'='*60}
#{len(query_history) - i + 1} | {entry['timestamp']}
Question: {entry['question'][:80]}...
SQL: {entry['sql'][:100]}...
{'='*60}

"""
    return history_text

def clear_history():
    """Clear all history"""
    global query_history
    query_history = []
    return "✅ History cleared successfully!"

# Database schema presets
schema_presets = {
    "🛒 E-commerce": """CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    name VARCHAR(100),
    email VARCHAR(100),
    registration_date DATE
);

CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(100),
    category VARCHAR(50),
    price DECIMAL(10,2),
    stock_quantity INT
);

CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    order_date DATE,
    total_amount DECIMAL(10,2),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);""",

    "👥 Employee Management": """CREATE TABLE employees (
    emp_id INT PRIMARY KEY,
    name VARCHAR(100),
    department VARCHAR(50),
    position VARCHAR(50),
    salary DECIMAL(10,2),
    hire_date DATE,
    manager_id INT
);

CREATE TABLE departments (
    dept_id INT PRIMARY KEY,
    dept_name VARCHAR(50),
    location VARCHAR(100),
    budget DECIMAL(12,2)
);

CREATE TABLE projects (
    project_id INT PRIMARY KEY,
    project_name VARCHAR(100),
    start_date DATE,
    end_date DATE
);""",

    "🎓 School": """CREATE TABLE students (
    student_id INT PRIMARY KEY,
    name VARCHAR(100),
    major VARCHAR(50),
    gpa DECIMAL(3,2),
    enrollment_date DATE
);

CREATE TABLE courses (
    course_id INT PRIMARY KEY,
    course_name VARCHAR(100),
    credits INT,
    department VARCHAR(50)
);

CREATE TABLE enrollments (
    enrollment_id INT PRIMARY KEY,
    student_id INT,
    course_id INT,
    semester VARCHAR(20),
    grade VARCHAR(2)
);""",

    "🏥 Hospital": """CREATE TABLE patients (
    patient_id INT PRIMARY KEY,
    name VARCHAR(100),
    date_of_birth DATE,
    phone VARCHAR(20)
);

CREATE TABLE doctors (
    doctor_id INT PRIMARY KEY,
    name VARCHAR(100),
    specialization VARCHAR(50),
    department VARCHAR(50)
);

CREATE TABLE appointments (
    appointment_id INT PRIMARY KEY,
    patient_id INT,
    doctor_id INT,
    appointment_date DATETIME,
    status VARCHAR(20)
);"""
}

# Sample questions for each preset
sample_questions = {
    "🛒 E-commerce": [
        "Show top 10 customers by total purchase amount",
        "Find products with stock less than 10 units",
        "Calculate average order value per month",
        "List customers who haven't ordered in 90 days"
    ],
    "👥 Employee Management": [
        "Find average salary by department",
        "Show employees hired in last 6 months",
        "List all employees and their managers",
        "Find top 5 highest paid employees"
    ],
    "🎓 School": [
        "Show students with GPA above 3.5",
        "List courses with most enrollments",
        "Find students who failed any course",
        "Calculate average GPA by major"
    ],
    "🏥 Hospital": [
        "Show appointments for next week",
        "Find doctors with most appointments",
        "List patients with multiple visits",
        "Show today's appointment schedule"
    ]
}

# ==================== CREATE GRADIO UI ====================

custom_css = """
.gradio-container {
    font-family: 'Inter', sans-serif;
    max-width: 1400px !important;
}
.header-text {
    text-align: center;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 2.5em;
    font-weight: bold;
    margin-bottom: 0.5em;
}
.sub-header {
    text-align: center;
    color: #666;
    font-size: 1.2em;
    margin-bottom: 2em;
}
.output-box {
    border-left: 4px solid #667eea;
    padding-left: 10px;
}
.status-success {
    color: #10b981;
    font-weight: bold;
}
.stat-card {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    padding: 20px;
    border-radius: 10px;
    color: white;
    text-align: center;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft(), title="SQL Query Generator") as demo:

    # Header
    gr.HTML("""
        <div class="header-text">
            🔍 AI-Powered SQL Query Generator
        </div>
        <div class="sub-header">
            Transform natural language into SQL queries instantly using Fine-tuned Mistral-7B
        </div>
    """)

    with gr.Tabs() as tabs:

        # ==================== TAB 1: QUERY GENERATOR ====================
        with gr.Tab("🚀 Generate Query", id=0):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 📊 Database Schema")

                    schema_dropdown = gr.Dropdown(
                        choices=["Custom"] + list(schema_presets.keys()),
                        value="Custom",
                        label="Choose Schema Preset",
                        info="Select a preset or enter custom schema"
                    )

                    schema_input = gr.Textbox(
                        lines=12,
                        placeholder="CREATE TABLE users (id INT, name TEXT, age INT);",
                        label="",
                        show_label=False
                    )

                    gr.Markdown("### ❓ Your Question")

                    question_dropdown = gr.Dropdown(
                        choices=[],
                        label="Sample Questions",
                        info="Select a sample or write your own",
                        visible=False,
                        interactive=True
                    )

                    question_input = gr.Textbox(
                        lines=3,
                        placeholder="What would you like to query?",
                        label="",
                        show_label=False
                    )

                    with gr.Row():
                        generate_btn = gr.Button("✨ Generate SQL", variant="primary", size="lg", scale=2)
                        clear_btn = gr.Button("🗑️ Clear", size="lg", scale=1)

                with gr.Column(scale=1):
                    gr.Markdown("### ✨ Generated Output")

                    status_text = gr.Textbox(
                        label="Status",
                        value="Ready to generate...",
                        interactive=False,
                        show_label=False
                    )

                    sql_output = gr.Textbox(
                        lines=10,
                        label="SQL Query",
                        show_copy_button=True,
                        show_label=True
                    )

                    explanation_output = gr.Textbox(
                        lines=6,
                        label="Explanation",
                        show_label=True
                    )

                    gr.Markdown("### 💾 Save Options")
                    with gr.Row():
                        save_current_btn = gr.Button("💾 Save Current Query", size="sm")
                        download_current = gr.File(label="Download", visible=False)

            # Sample examples section
            gr.Markdown("### 📚 Example Queries")
            gr.Examples(
                examples=[
                    [schema_presets["🛒 E-commerce"], "Show the top 5 customers by total purchase amount"],
                    [schema_presets["👥 Employee Management"], "Find the average salary for each department"],
                    [schema_presets["🎓 School"], "List all students with GPA higher than 3.5"],
                    [schema_presets["🏥 Hospital"], "Show all appointments scheduled for tomorrow"],
                ],
                inputs=[schema_input, question_input],
                label="Try these examples"
            )

        # ==================== TAB 2: QUERY HISTORY ====================
        with gr.Tab("📜 Query History", id=1):
            gr.Markdown("### 📊 Your Query History")

            with gr.Row():
                with gr.Column(scale=3):
                    history_display = gr.Textbox(
                        lines=20,
                        label="Recent Queries (Last 10)",
                        value="No queries generated yet.",
                        interactive=False
                    )

                with gr.Column(scale=1):
                    gr.Markdown("### 📈 Statistics")
                    total_queries = gr.Markdown("**Total Queries:** 0")

                    gr.Markdown("### 💾 Export Options")

                    refresh_btn = gr.Button("🔄 Refresh History", variant="secondary")

                    with gr.Row():
                        export_json_btn = gr.Button("📄 Export JSON", size="sm")
                        export_csv_btn = gr.Button("📊 Export CSV", size="sm")

                    download_json = gr.File(label="Download JSON")
                    download_csv = gr.File(label="Download CSV")

                    gr.Markdown("---")

                    clear_history_btn = gr.Button("🗑️ Clear History", variant="stop")
                    clear_status = gr.Textbox(label="Status", visible=False)

        # ==================== TAB 3: HELP & INFO ====================
        with gr.Tab("ℹ️ Help & Info", id=2):
            gr.Markdown("""
            # 📖 User Guide

            ## 🚀 How to Use

            1. **Select or Enter Schema**: Choose a preset schema or enter your custom database schema
            2. **Ask Your Question**: Type your question in natural language
            3. **Generate SQL**: Click the "Generate SQL" button
            4. **Review & Save**: Copy the SQL query or save it to your local machine

            ## 💾 Saving Options

            ### Save Current Query
            - Saves the current query to a timestamped `.txt` file
            - Files are saved in: `sql_generator_outputs/query_TIMESTAMP.txt`

            ### Export History
            - **JSON Export**: All queries in structured JSON format
            - **CSV Export**: All queries in spreadsheet-compatible CSV format
            - Files are saved in: `sql_generator_outputs/query_history_TIMESTAMP.json/csv`

            ## 📂 Output Location

            All files are saved to: **`sql_generator_outputs/`** directory in your current working folder

            ## 🎯 Tips for Best Results

            - Provide complete table schemas with column types
            - Be specific in your questions
            - Use proper table and column names
            - For complex queries, break them into steps

            ## 🔧 Model Information

            - **Base Model**: Mistral-7B-Instruct-v0.2
            - **Fine-tuning**: Custom SQL generation dataset
            - **Technique**: LoRA (Low-Rank Adaptation)
            - **Precision**: 4-bit quantization

            ## 📊 Supported Query Types

            ✅ SELECT statements
            ✅ JOIN operations (INNER, LEFT, RIGHT)
            ✅ Aggregations (COUNT, SUM, AVG, etc.)
            ✅ GROUP BY and HAVING clauses
            ✅ Subqueries
            ✅ Date/Time filtering
            ✅ ORDER BY and LIMIT

            ## ⚠️ Limitations

            - Complex nested queries may require refinement
            - Database-specific syntax variations
            - Always test queries in a safe environment first

            ## 🆘 Need Help?

            If you encounter issues:
            1. Check your schema syntax
            2. Simplify your question
            3. Try the example queries first
            4. Review the generated explanation
            """)

    # ==================== EVENT HANDLERS ====================

    # Schema preset change
    def update_schema_and_questions(preset):
        if preset == "Custom":
            return "", gr.Dropdown(choices=[], visible=False)
        else:
            schema = schema_presets.get(preset, "")
            questions = sample_questions.get(preset, [])
            return schema, gr.Dropdown(choices=questions, visible=True, value=None)

    schema_dropdown.change(
        fn=update_schema_and_questions,
        inputs=[schema_dropdown],
        outputs=[schema_input, question_dropdown]
    )

    # Question preset selection
    def update_question(question):
        return question if question else ""

    question_dropdown.change(
        fn=update_question,
        inputs=[question_dropdown],
        outputs=[question_input]
    )

    # Generate SQL
    generate_btn.click(
        fn=generate_sql_response,
        inputs=[schema_input, question_input],
        outputs=[sql_output, explanation_output, status_text]
    )

    # Clear inputs
    def clear_inputs():
        return "", "", "", "", "Ready to generate..."

    clear_btn.click(
        fn=clear_inputs,
        inputs=[],
        outputs=[schema_input, question_input, sql_output, explanation_output, status_text]
    )

    # Save current query
    def save_and_return(schema, question, sql, explanation):
        if not sql or sql == "":
            return None
        filename = save_current_query(schema, question, sql, explanation)
        return gr.File(value=filename, visible=True)

    save_current_btn.click(
        fn=save_and_return,
        inputs=[schema_input, question_input, sql_output, explanation_output],
        outputs=[download_current]
    )

    # Refresh history
    def refresh_history_display():
        history_text = get_history_display()
        stats = f"**Total Queries:** {len(query_history)}"
        return history_text, stats

    refresh_btn.click(
        fn=refresh_history_display,
        inputs=[],
        outputs=[history_display, total_queries]
    )

    # Export JSON
    def export_json_handler():
        if not query_history:
            return None
        filename = save_to_json()
        return gr.File(value=filename, visible=True)

    export_json_btn.click(
        fn=export_json_handler,
        inputs=[],
        outputs=[download_json]
    )

    # Export CSV
    def export_csv_handler():
        if not query_history:
            return None
        filename = save_to_csv()
        return gr.File(value=filename, visible=True)

    export_csv_btn.click(
        fn=export_csv_handler,
        inputs=[],
        outputs=[download_csv]
    )

    # Clear history
    clear_history_btn.click(
        fn=clear_history,
        inputs=[],
        outputs=[clear_status]
    ).then(
        fn=refresh_history_display,
        inputs=[],
        outputs=[history_display, total_queries]
    )

print("=" * 70)
print("🚀 LAUNCHING ENHANCED SQL QUERY GENERATOR")
print("=" * 70)
print(f"📂 Output directory: {os.path.abspath(output_dir)}")
print("🌐 Interface will open in your browser")
print("🔗 A shareable link will be generated")
print("=" * 70)

demo.launch(share=True, debug=True)


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


✅ Using merged model
🚀 LAUNCHING ENHANCED SQL QUERY GENERATOR
📂 Output directory: /content/sql_generator_outputs
🌐 Interface will open in your browser
🔗 A shareable link will be generated
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6a8f1c771242e728ae.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
